# 音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜

### 第13章：研究デザインと検定力：シミュレーションで決める


ここに掲載されているコードは、
『音声研究のための統計解析 〜RとColaboratoryで学ぶ実践ハンドブック〜』の付録である。
本書の巻末に載っているGitHubリポジトリのリンクから、「Open in Colab」のバッジで開いている。

「このノートブックはGoogleが作成したものではありません」という警告は、外部（GitHub）から
読み込むノートブック全般に出る定型の警告である。内容を確認のうえ実行してよい。
書き込んだ内容を残したいときは、「ファイル」→「ドライブにコピーを保存」で自分のGoogle Driveに
保存する。保存したコピーは自由に変更・実行・保存してよい。

**説明はすべてコード中の#コメントとセル内のmarkdownに書いてある。**
コピペ元を別に参照する必要はなく、このノートブックを上から実行しながら読めば完結する。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で言語を **R** に切り替えること。
既定のPythonランタイムのままだと以降のセルがすべてエラーになる。


参照: Brysbaert & Stevens (2018) 混合効果モデルの検定力／Hoenig & Heisey (2001) 事後検定力の誤用／Westfall, Kenny & Judd (2014) 交差ランダム効果デザインの検定力。

In [ ]:
install.packages(c("lme4", "lmerTest"), repos = "https://cloud.r-project.org")
library(lme4)
library(lmerTest)
library(tidyverse)
# ---- 図の日本語文字化け対策・文字サイズの既定値 ----
# Colabの既定フォントには日本語グリフが含まれないため、図のタイトルや軸ラベルが
# 文字化け（豆腐記号）して表示される。Google Fontsの日本語フォントを取得し、
# "notosans"という名前で登録して使う（要ネット接続。Colabでは常に使える）。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")  # "sans"などの予約名は上書きできないため専用名で登録
showtext_auto()
showtext_opts(dpi = 100)                     # Colabのインライン画像解像度(100dpi)に合わせる
par(family = "notosans")                     # base Rの作図（mclust・itsadugなど）にも適用する
theme_set(theme_minimal(base_size = 14, base_family = "notosans"))  # ggplotの既定テーマ


■ 「何人録ればいいですか」に答える：

研究を始める前に必ず出てくる問いが、**話者を何人、語を何語用意すればよいか**である。この問いに勘で答えると、2つの失敗のどちらかが起きる。少なすぎれば、効果が実在しても検出できずに「差は無かった」と書くことになる。多すぎれば、実質的に意味のない差まで有意になり、録音と分析にかけた労力も無駄になる。

混合効果モデルの検定力は、教科書の公式では計算できない。話者のランダム切片・ランダム傾き・項目のランダム切片・残差という複数の分散が絡むためである。そこで実務では**シミュレーション**で求める。考え方は単純で、「想定する条件でデータを何百組も作り、そのうち何割で有意になるか」を数えるだけである。

■ 検定力を「当てはまる割合」として数える：

効果が本当にある世界で実験を2000回繰り返し、有意になった割合を数える。単純な2群比較ではこの割合が理論値（power.t.test）とほぼ一致する。混合効果モデルでは理論式が使いにくいので、同じ発想で次のシミュレーションを行う。

In [ ]:
# 2群比較の検定力を、シミュレーションと理論式で並べる
sim_power_t <- function(n, delta, sd = 15, nsim = 2000, seed = 13) {
  set.seed(seed)
  p <- replicate(nsim, {
    a <- rnorm(n, 0, sd)
    b <- rnorm(n, delta, sd)
    t.test(a, b, var.equal = TRUE)$p.value
  })
  mean(p < 0.05)
}
res <- expand.grid(n = c(10, 20, 40, 80), delta = c(5, 10))
res$simulation <- mapply(sim_power_t, res$n, res$delta)
res$theory <- mapply(function(n, d)
  power.t.test(n = n, delta = d, sd = 15, sig.level = 0.05)$power,
  res$n, res$delta)
res$simulation <- round(res$simulation, 3)
res$theory <- round(res$theory, 3)
print(res, row.names = FALSE)

# --- 実行結果 ---
#   n delta simulation theory
#  10     5      0.102  0.105
#  20     5      0.165  0.176
#  40     5      0.318  0.313
#  80     5      0.566  0.554
#  10    10      0.294  0.292
#  20    10      0.538  0.538
#  40    10      0.841  0.838
#  80    10      0.987  0.987
#
# 【出力の読み方】
# simulation（2000回のt検定でp < .05になった割合）は theory（power.t.testの理論値）とほぼ一致し、
# 最大の差は0.012である。差5 msでは各群80でも検定力は約.55、差10 msなら各群40で約.84に届く。
# 検定力は標本サイズ・効果量・ばらつきの3つで決まる。


■ 検定力を計算する関数を作る：

In [ ]:
# 引数で「想定する世界」を指定し、その世界で何割の実験が成功するかを返す。
# effect  : 検出したい効果の大きさ（ms）
# sd_sp   : 話者ごとの全体水準のばらつき
# sd_slope: 話者ごとの「効果の大きさ」自体のばらつき（ランダム傾き）
# sd_it   : 項目ごとのばらつき
# sd_res  : 残差
simulate_power <- function(n_speaker, n_item, effect,
                           sd_sp = 8, sd_slope = 6, sd_it = 4, sd_res = 7,
                           nsim = 200, seed = 1) {
  set.seed(seed)
  # 条件は「語の性質」として割り当てる（第1章のvoiced/voicelessと同じ設計）
  cond_of_item <- rep(c(-0.5, 0.5), length.out = n_item)

  out <- replicate(nsim, {
    sp_int   <- rnorm(n_speaker, 0, sd_sp)
    sp_slope <- rnorm(n_speaker, 0, sd_slope)
    it_int   <- rnorm(n_item,    0, sd_it)

    d <- expand.grid(sp = 1:n_speaker, it = 1:n_item)
    d$cond <- cond_of_item[d$it]
    d$y <- 50 + effect * d$cond +
           sp_int[d$sp] + sp_slope[d$sp] * d$cond +
           it_int[d$it] + rnorm(nrow(d), 0, sd_res)
    d$sp <- factor(d$sp); d$it <- factor(d$it)

    # ★収束の警告・singular fitを黙って揉み消さない。異常フィットを
    #   別カウントし、p値だけを見て成功/失敗をカウントすることの危険を避ける。
    anomaly <- FALSE
    m <- withCallingHandlers(
      tryCatch(lmer(y ~ cond + (1 + cond | sp) + (1 | it), data = d),
               error = function(e) NULL),
      warning = function(w) { anomaly <<- TRUE; invokeRestart("muffleWarning") }
    )
    if (is.null(m)) return(c(sig = NA, anomaly = 1))
    if (isSingular(m)) anomaly <- TRUE
    sig <- coef(summary(m))["cond", "Pr(>|t|)"] < 0.05
    c(sig = as.numeric(sig), anomaly = as.numeric(anomaly))
  })

  sig_vals <- out["sig", ]
  anomaly_vals <- out["anomaly", ]
  n_valid <- sum(!is.na(sig_vals))
  n_success <- sum(sig_vals, na.rm = TRUE)
  # ★プラグインSE（sqrt(p(1-p)/n)）は境界（p=0やp=1）で0になり、
  #   「不確実性ゼロ」という誤った印象を与える（200回中200回成功しても、
  #   真の検定力が厳密に1だとは言えない）。二項検定の信頼区間
  #   （Clopper-Pearson）を使えば、境界でも幅のある区間が出る。
  ci <- binom.test(n_success, n_valid)$conf.int
  # 収束の警告・singular fitを起こさなかった試行だけに絞った検定力も
  # 別に計算する。anomalyを起こした試行のp値は信用できないため。
  clean_idx <- !is.na(sig_vals) & anomaly_vals == 0
  power_clean <- if (sum(clean_idx) > 0) mean(sig_vals[clean_idx]) else NA
  list(power = n_success / n_valid, ci_lo = ci[1], ci_hi = ci[2],
       anomaly_rate = mean(anomaly_vals), n_valid = n_valid,
       power_clean = power_clean, n_clean = sum(clean_idx))
}

# ★健全性チェック: 各話者が両方の条件を持っているかを必ず確認する。
#   ここを間違えると（条件が話者と交絡すると）、検定力の見積もりが丸ごと狂う。
check <- expand.grid(sp = 1:4, it = 1:6)
check$cond <- rep(c(-0.5, 0.5), length.out = 6)[check$it]
cat("話者ごとに現れる条件の種類数:",
    unique(tapply(check$cond, check$sp, function(x) length(unique(x)))), "\n")

# 結果を「検定力 [95%信頼区間], 異常フィット率」の形で表示する補助関数。
fmt_power <- function(r) sprintf("%.2f [95%%CI %.2f-%.2f], 異常フィット率 %.0f%%",
                                  r$power, r$ci_lo, r$ci_hi, r$anomaly_rate * 100)


■ 話者を増やすべきか、語を増やすべきか：

録音の手間は話者を増やすほうが圧倒的に大きい。同じ検定力を得られるなら、語を増やすほうが安上がりである。どちらが効くかを実際に計算して確かめる。

In [ ]:
# ⏳ 各設定で200回モデルを当てはめるので、このセルは数分かかる。
cat("【設計A】条件が「語の性質」である場合（voiced語 vs voiceless語）\n")
cat("  効果5ms、項目20語に固定して話者数を変える:\n")
for (n in c(10, 20, 40)) {
  cat(sprintf("    話者%2d名: 検定力 %s\n", n, fmt_power(simulate_power(n, 20, effect = 5))))
}
cat("  効果5ms、話者20名に固定して項目数を変える:\n")
for (k in c(10, 20, 40)) {
  cat(sprintf("    項目%2d語: 検定力 %s\n", k, fmt_power(simulate_power(20, k, effect = 5))))
}

# --- 実行結果 ---
# 話者数を変えた側（効果5ms・項目20語固定）:
#   話者10名: 検定力 0.41 [95%CI 0.34-0.48], 異常フィット率 1%
#   話者20名: 検定力 0.57 [95%CI 0.50-0.64], 異常フィット率 0%
#   話者40名: 検定力 0.64 [95%CI 0.57-0.71], 異常フィット率 0%
# 項目数を変えた側（効果5ms・話者20名固定）:
#   項目10語: 検定力 0.41 [95%CI 0.35-0.49], 異常フィット率 1%
#   項目20語: 検定力 0.57 [95%CI 0.50-0.64], 異常フィット率 0%
#   項目40語: 検定力 0.74 [95%CI 0.67-0.80], 異常フィット率 1%
#
# 【出力の読み方】
# ・検定力とは「効果が本当にあるとき、それを有意として検出できる確率」である。
#   慣習的な目標は0.80で、5回に4回は成功する設計を意味する。
# ・**表示には二項検定の95%信頼区間と異常フィット率を必ず併記する。**
#   nsim=200程度のシミュレーションでは検定力0.5〜0.6付近で区間の幅が
#   7ポイント前後にもなるため、検定力を3桁目まで表示するのは過剰な精度である。
#   ★プラグインのSE（√(p(1-p)/n)）は境界（検定力0や1）で0になり、
#   「不確実性ゼロ」という誤った印象を与える。二項検定の信頼区間なら、
#   境界でも意味のある幅が出る（後の演習で200/200成功の例を扱う）。
# ・**異常フィット率（収束警告・singular fitの起きた割合）も検定力と同時に見る。**
#   これが高ければ、その設定でのlmer()の当てはめ自体が不安定であることを
#   意味し、検定力の推定値も信用できない。0%に近ければ問題ないが、
#   高い場合は`sd_slope`を下げる、`nsim`を増やす、あるいはモデルの構造
#   そのものを見直す必要がある（後の演習3で、この率が跳ね上がる例を扱う）。
# ・話者数・項目数のどちらを増やすほうが効くかは、条件が語の性質そのもので
#   あるという設計（設計A）に固有の性質から決まる。次のセルで、
#   条件の入れ方を変える設計Bと比べる。


■ 設計そのものを変えると何が起きるか：

設計Aでは、voiced語とvoiceless語という**別々の語**を比べていた。これを、**同じ語を2つの条件で発話してもらう**設計に変えるとどうなるか。たとえば「同じ単語を、丁寧な発話とくだけた発話で読んでもらう」ような場合である。

In [ ]:
# 設計B: 条件を項目の内側で操作する（同じ語が両条件に現れる）
simulate_power_B <- function(n_speaker, n_item, effect,
                             sd_sp = 8, sd_slope = 6, sd_it = 4, sd_res = 7,
                             nsim = 200, seed = 2) {
  set.seed(seed)
  out <- replicate(nsim, {
    sp_int   <- rnorm(n_speaker, 0, sd_sp)
    sp_slope <- rnorm(n_speaker, 0, sd_slope)
    it_int   <- rnorm(n_item,    0, sd_it)
    # cond を設計に含めるので、同じ語が両条件に現れる
    d <- expand.grid(sp = 1:n_speaker, it = 1:n_item, cond = c(-0.5, 0.5))
    d$y <- 50 + effect * d$cond +
           sp_int[d$sp] + sp_slope[d$sp] * d$cond +
           it_int[d$it] + rnorm(nrow(d), 0, sd_res)
    d$sp <- factor(d$sp); d$it <- factor(d$it)
    # ★設計Aと同じく、異常フィットを黙って揉み消さない。
    anomaly <- FALSE
    m <- withCallingHandlers(
      tryCatch(lmer(y ~ cond + (1 + cond | sp) + (1 | it), data = d),
               error = function(e) NULL),
      warning = function(w) { anomaly <<- TRUE; invokeRestart("muffleWarning") }
    )
    if (is.null(m)) return(c(sig = NA, anomaly = 1))
    if (isSingular(m)) anomaly <- TRUE
    sig <- coef(summary(m))["cond", "Pr(>|t|)"] < 0.05
    c(sig = as.numeric(sig), anomaly = as.numeric(anomaly))
  })
  sig_vals <- out["sig", ]
  n_valid <- sum(!is.na(sig_vals))
  n_success <- sum(sig_vals, na.rm = TRUE)
  ci <- binom.test(n_success, n_valid)$conf.int  # 設計Aと同じ理由でCIを使う
  list(power = n_success / n_valid, ci_lo = ci[1], ci_hi = ci[2],
       anomaly_rate = mean(out["anomaly", ]), n_valid = n_valid)
}

cat("【設計B】同じ語を両条件で発話してもらう場合\n")
cat("  効果5ms、項目20語に固定して話者数を変える:\n")
for (n in c(10, 20, 40)) {
  cat(sprintf("    話者%2d名: 検定力 %s\n", n, fmt_power(simulate_power_B(n, 20, effect = 5))))
}
cat("  効果5ms、話者20名に固定して項目数を変える:\n")
for (k in c(10, 20, 40)) {
  cat(sprintf("    項目%2d語: 検定力 %s\n", k, fmt_power(simulate_power_B(20, k, effect = 5))))
}

# --- 実行結果 ---
# 話者数を変えた側（効果5ms・項目20語固定）:
#   話者10名: 検定力 0.62 [95%CI 0.55-0.69], 異常フィット率 0%
#   話者20名: 検定力 0.91 [95%CI 0.86-0.95], 異常フィット率 0%
#   話者40名: 検定力 1.00 [95%CI 0.98-1.00], 異常フィット率 0%
# 項目数を変えた側（効果5ms・話者20名固定）:
#   項目10語: 検定力 0.86 [95%CI 0.81-0.91], 異常フィット率 0%
#   項目20語: 検定力 0.91 [95%CI 0.86-0.95], 異常フィット率 0%
#   項目40語: 検定力 0.95 [95%CI 0.92-0.98], 異常フィット率 0%
#
# 【出力の読み方】
# ・**話者40名の検定力1.00は「200回中200回成功した」という意味であり、
#   95%信頼区間は[0.98, 1.00]である。「真の検定力が厳密に1」という意味では
#   ない。** 200回では区別できないだけで、本当は0.99かもしれない。
#   プラグインのSEを使うとこの境界で誤って「SE = 0（不確実性ゼロ）」と
#   表示されてしまうため、二項検定の信頼区間を使う。
# ・項目数を20語のまま揃えて確認できる範囲では、設計Bの検定力（0.91前後）は
#   設計Aの同条件（0.57）よりはっきり高い。理由は明快である。設計Bでは
#   同じ語が両条件に現れるので、**語ごとのばらつきが条件の比較から
#   差し引かれる**。語Aが本来長めであっても、条件1でも条件2でも同じだけ
#   長いので、その差を取れば消える。設計Aではこれができなかった。
# ・★実務上の結論は3つある。
#   (1) **可能なら、条件は項目の内側で操作する設計にする。**
#       同じ語を条件をまたいで使えるなら、それだけで検定力が大きく上がる。
#   (2) **設計が決まってから、増やすべき単位を決める。**
#       「話者を増やせばよい」「項目を増やせばよい」という一般則は無い。
#   (3) **どちらの設計でも、検定力はシミュレーションで確かめる。** 勘では
#       当てられない。異常フィット率も一緒に確認し、高ければ結果を疑う。
# ・★設計Aが悪いわけではない。voiced対voicelessのように、
#   **条件が語の性質そのものである場合は、設計Bは原理的に不可能**である。
#   そのときは、語数を十分に確保することが決定的に重要になる。


■ ★よくある誤り：実験のあとに検定力を計算する（事後検定力）：

「有意にならなかったので、検定力を計算したら0.3しかなかった。だからサンプル不足が原因である」——この推論は、**観測された効果量を使って計算する限り、何も情報を加えていない**。観測された効果量から計算した検定力はp値の単調な関数にすぎず、p値が大きければ必ず低く出るからである。

In [ ]:
# 観測された効果量を使った「事後検定力」が、p値と一対一に対応することを確かめる。
set.seed(88)
results <- map_dfr(1:60, function(i) {
  n <- 20
  x <- rep(c(-0.5, 0.5), length.out = n)
  y <- 0.4 * x + rnorm(n)          # 真の効果は 0.4 で固定
  tt <- t.test(y ~ factor(x))
  d_obs <- diff(rev(tapply(y, x, mean))) / sd(y)   # 観測された標準化効果量
  # 観測された効果量をそのまま使って計算した「事後検定力」
  pw <- power.t.test(n = n / 2, delta = abs(d_obs), sd = 1, sig.level = 0.05)$power
  tibble(p = tt$p.value, post_hoc_power = pw)
})

cat("p値と事後検定力の相関:", round(cor(results$p, results$post_hoc_power), 4), "\n")
cat("（順位相関）           :",
    round(cor(results$p, results$post_hoc_power, method = "spearman"), 4), "\n\n")
print(results %>% arrange(p) %>% slice(c(1, 2, 30, 59, 60)) %>%
        mutate(across(everything(), ~round(.x, 4))))

# --- 実行結果 ---
# p値と事後検定力の相関: -0.9021
# （順位相関）           : -0.9998
#
# # A tibble: 5 x 2
#        p post_hoc_power
#    <dbl>          <dbl>
# 1 0.0339         0.521
# 2 0.0358         0.497
# 3 0.312          0.164
# 4 0.922          0.0312
# 5 0.949          0.029
#
# 【出力の読み方】
# ・**順位相関が −0.9998 と、ほぼ完全に −1 である。** つまりp値が大きいほど
#   事後検定力は必ず小さくなり、両者の対応は事実上1対1に決まっている。
#   p = 0.0339 のとき検定力0.52、p = 0.949 のとき検定力0.029という具合である。
#   （ちょうど−1にならないのは、効果量の標準化に全体のSDを使っているためで、
#   検定が使う群内の統合SDとわずかにずれる。本質的な違いではない。）
# ・したがって「p値は有意でなかった。事後検定力を計算したら低かった。
#   だからサンプル不足だ」という主張は、**「p値が大きかったので、p値が
#   大きかった」と言っているのと同じ**である。新しい情報は1つも無い。
# ・★では有意にならなかったとき、どう報告すべきか。答えは
#   **信頼区間を示すこと**である。区間が狭くて0の近くにあるなら
#   「効果は小さい」と言え、区間が広いなら「まだ分からない」と言える。
#   この2つを区別できるのは信頼区間だけで、p値にも事後検定力にもできない。
# ・★検定力の計算が意味を持つのは、**実験を計画する段階で、
#   検出したい効果量を自分で決めて**行う場合だけである。
#   「先行研究の効果量を使えばよい」と考えたくなるが、これにも罠がある。
#   出版された研究は有意になったものに偏っているため、報告された効果量は
#   真の値より**大きめに出ている**（winner's curse）。
#   先行研究の効果量をそのまま使うと、必要な規模を過小に見積もることになる。
#   **「これ以上小さければ実質的に意味が無い」と自分で決めた値**を使うのが、
#   最も防御しやすい考え方である。


■ winner's curse の大きさを数値で見る（Type M・Type S error）：

検定力が低いとき、有意になった推定値が真値をどれだけ誇張するか（Type M error、誇張率）と、有意でも符号が逆になる確率（Type S error）を、Gelman & Carlin (2014) の考え方で計算する。

In [ ]:
retro <- function(D, s, alpha = 0.05, n = 1e6) {
  z <- qnorm(1 - alpha / 2)
  set.seed(1)
  est <- rnorm(n, D, s)               # 真の効果D、標準誤差sのもとでの推定値の分布
  sig <- abs(est / s) > z             # 有意になった推定値だけを選ぶ
  c(power = mean(sig),
    typeS = mean(sign(est[sig]) != sign(D)),
    typeM = mean(abs(est[sig])) / D)
}
for (s in c(1.5, 3, 5)) print(round(c(se = s, retro(5, s)), 3))

# --- 実行結果 ---
#   se power typeS typeM
#  1.5 0.915 0.000 1.051
#  3.0 0.385 0.000 1.596
#  5.0 0.170 0.009 2.491
#
# 【出力の読み方】
# ・標準誤差1.5ms（検定力0.915）なら、有意になった推定値の平均は真値の1.05倍で
#   ほぼ誇張されない。
# ・標準誤差5ms（検定力0.17）まで下がると、**有意になった推定値は平均して真値の
#   約2.5倍に誇張**され、約1%は符号まで逆になる。
# ・つまり検定力は「有意になる確率」であると同時に、「有意になったときの推定値が
#   どれだけ信頼できるか」を決める量でもある。先行研究の効果量をそのまま計画に
#   使ってはいけない理由の数値的な裏付けである。


■ 演習：

1. 設計Aで、効果を5msから10msに変えたときの検定力を計算せよ。
2. 設計Aで検定力0.80を達成するには、話者20名のとき項目が何語必要か。
3. `sd_slope`（話者ごとの効果のばらつき）を6から2に下げると、検定力がどう変わるか確かめ、その理由を説明せよ。

■ 演習の解答：

In [ ]:
# 1. 効果が2倍になると検定力はどうなるか。
cat("設計A・話者20名・項目20語:\n")
cat("  効果 5ms:", fmt_power(simulate_power(20, 20, effect = 5)), "\n")
cat("  効果10ms:", fmt_power(simulate_power(20, 20, effect = 10)), "\n")

# 2. 項目数を増やして0.80を超える点を探す。
cat("\n設計A・話者20名・効果5ms で項目数を増やす:\n")
for (k in c(40, 60, 80, 120)) {
  cat(sprintf("  項目%3d語: 検定力 %s\n", k, fmt_power(simulate_power(20, k, effect = 5))))
}

# 3. 話者ごとの効果のばらつきを小さくする。
cat("\n設計A・話者20名・項目20語・効果5ms:\n")
r_slope6 <- simulate_power(20, 20, 5, sd_slope = 6)
r_slope2 <- simulate_power(20, 20, 5, sd_slope = 2)
cat("  sd_slope = 6:", fmt_power(r_slope6), "\n")
cat("  sd_slope = 2:", fmt_power(r_slope2), "\n")
cat("  sd_slope = 2の内訳: 異常フィットを除いた", r_slope2$n_clean, "/", r_slope2$n_valid,
    "回だけで検定力を計算すると", round(r_slope2$power_clean, 3), "\n")

# --- 実行結果 ---
# 設計A・話者20名・項目20語:
#   効果 5ms: 0.57 [95%CI 0.50-0.64], 異常フィット率 0%
#   効果10ms: 0.98 [95%CI 0.95-0.99], 異常フィット率 0%
#
# 設計A・話者20名・効果5ms で項目数を増やす:
#   項目 40語: 検定力 0.74 [95%CI 0.67-0.80], 異常フィット率 1%
#   項目 60語: 検定力 0.78 [95%CI 0.71-0.83], 異常フィット率 0%
#   項目 80語: 検定力 0.83 [95%CI 0.78-0.88], 異常フィット率 0%
#   項目120語: 検定力 0.86 [95%CI 0.81-0.91], 異常フィット率 0%
#
# 設計A・話者20名・項目20語・効果5ms:
#   sd_slope = 6: 0.57 [95%CI 0.50-0.64], 異常フィット率 0%
#   sd_slope = 2: 0.73 [95%CI 0.66-0.79], 異常フィット率 22%
#   sd_slope = 2の内訳: 異常フィットを除いた155/200回だけで検定力を計算すると0.723
#
# 【出力の読み方】
# ・1問目: 効果が5msから10msへ2倍になると、検定力は0.57から0.98へ跳ね上がる。
#   **検定力は効果量に対して急激に反応する。** だからこそ、
#   「どれくらいの効果を検出したいか」を決めることが、
#   サンプルサイズの計算そのものより重要になる。
#   ★ここを「先行研究では10msだったから10msで計算しよう」と済ませると、
#   実際の効果が5msだったときに設計が破綻する。**検出したい最小の効果**
#   （これより小さければ実質的に意味がない、と言える値）で計算するのが安全である。
# ・2問目: 項目を40語から120語へ3倍にしても、検定力は0.74から0.86までしか
#   上がらない。**増やすほど効きが鈍る（収穫逓減）。** 0.80を超えるのは
#   80語あたりである。0.86から0.95へ上げるコストは、0.60から0.80へ上げる
#   コストよりはるかに大きい。**どこで止めるかは、統計ではなく研究の
#   予算と価値判断で決める。**
# ・3問目: 話者ごとの効果のばらつき（sd_slope）を6から2へ下げると、
#   検定力は0.57から0.73へ上がる——サンプルサイズは1つも変えていない。
#   **ただし、この設定（sd_slope = 2）では異常フィット率が22%まで跳ね上がる。**
#   200回中およそ45回で収束警告かsingular fitが起きている。sd_slopeを
#   小さくしすぎると、データの中に話者ごとの傾きのばらつきがほとんど無くなり、
#   モデルが「ランダム傾きの分散をほぼ0と推定する」ケースが増えるためである。
#   **この22%を無視して検定力0.73だけを報告するのは危険である。** 5回に
#   1回以上のペースで当てはめ自体が不安定な状況を含む数字だからである。
#   異常フィットを起こした45回を除いた「正常に収束した155回だけ」の検定力は
#   0.723と、全体の0.73とほぼ変わらない——**この設定では、異常フィットの
#   有無が検出のされやすさに強く偏っているわけではなかった。** ただし
#   一般には偏る可能性があるため、両方の数値を併記して読者に判断させるのが
#   安全である。異常フィット率が高い設定では、ランダム効果構造を簡略化する
#   （第5章参照）か、`nsim`を増やして異常フィットの影響を薄めるかを検討する。
# ・なぜsd_slopeを下げると検定力が上がるのか。検定したいのは「平均的な効果が
#   0か否か」だが、**話者ごとに効果の大きさが違えば、その平均の推定も
#   不安定になる**。全員がほぼ同じ大きさの効果を示すなら、少ない話者数でも
#   平均を精度よく決められる——ただし上で見たとおり、その代償として
#   モデルの当てはめ自体が不安定になりやすい場合がある。
# ・★実務的な含意が2つある。第一に、**均質な話者集団を選ぶと検定力が
#   上がりうる**。方言・年齢・性別を揃えれば効果のばらつきは減る。ただし
#   一般化できる範囲が狭まるだけでなく、モデルの当てはめそのものが
#   不安定になりうることも今回わかった。第二に、**録音条件を統制することも
#   検定力を上げる**。雑音や機材の違いは残差（sd_res）を膨らませ、
#   同じように検定力を削る。**サンプルサイズだけが検定力を決めるのではない。**
